# Chapter 13: the limits and the reconcile check, to compare with your report

Two known answers. The control chart: Mean FG 2.194 at fill 5 and at fill 35, flat limits at 1.81 and 2.58.
The waterfall: four stages add to 370 hL. If your Power BI report disagrees, fix the model first. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
b = load("Batch")
def mean_fg(at_fill, leak=False):
    rows = b[b.IsBaseline]
    if leak:                                                  # the axis filter reaching the measure
        rows = rows[rows.FillSequence == at_fill]
    return rows.FinalGravity.mean()                           # NaN here is a blank in Power BI
base = b[b.IsBaseline]
m, mr = base.FinalGravity.mean(), base.MovingRange.mean()
for f in (5, 35):
    print(f"fill {f}: Mean FG {mean_fg(f):.3f}  UCL {m + 2.66 * mr:.2f}  LCL {m - 2.66 * mr:.2f}  (with the filter leaking: {mean_fg(f, True):.3f})")
out = b[(b.FinalGravity > m + 2.66 * mr) | (b.FinalGravity < m - 2.66 * mr)]
print("Out of limits points:", len(out))

In [ ]:
st = load("Stage")
s = st[st.Month == "2026-03-01"].sort_values("StageOrder")
total = s.LossHL.sum()
print(s[["Stage", "LossHL"]].to_string(index=False))
print("total", total, "hL;", "PASS" if total == 370 else "FAIL", "| share of cast", round(100 * total / 8700, 2), "percent")
fig, ax = plt.subplots(figsize=(7, 3.5))
run = s.LossHL.cumsum()
ax.bar(s.Stage, s.LossHL, bottom=run - s.LossHL, color="tab:orange")
ax.bar(["Total"], [total], color="0.35"); ax.set_ylim(0, 400); ax.set_ylabel("Loss (hL)")
ax.set_title("Beer loss by stage, March 2026, hL", loc="left"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()